<a href="https://colab.research.google.com/github/bah862696-coder/DI-Bootcamp/blob/master/DayChallenge_week6_J6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Chargement et inspection des données

In [12]:
# Force reinstallation of specific, stable versions of datasets, huggingface_hub, numpy, and scipy
!pip install datasets==2.16.1 huggingface_hub==1.5.0 numpy==1.26.4 scipy==1.12.0 --force-reinstall -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2023.10.0 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires scipy>=1.13, but you have scipy 1.12.0 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires scipy>=1.13, but you have scipy 1.12.0 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.

In [4]:
import os
import shutil
from datasets import load_dataset

# Set environment variable to potentially workaround HfUriError
os.environ['HF_HUB_ENABLE_HF_SCAN'] = 'False'

# Clear Hugging Face cache to ensure fresh download and resolve potential issues
# This step is included again in case the reinstallation changed cache paths or behavior.
cache_dir = os.path.expanduser("~/.cache/huggingface")
if os.path.exists(cache_dir):
    print(f"Clearing Hugging Face cache at {cache_dir}")
    shutil.rmtree(cache_dir)
else:
    print(f"Hugging Face cache directory not found at {cache_dir}")

# Load the tweet_eval dataset with the 'sentiment' configuration
dataset = load_dataset('tweet_eval', 'sentiment')

# Display the data distribution (sizes of train, validation, test splits)
print("\nDataset splits distribution:")
for split, data in dataset.items():
    print(f"  {split}: {len(data)} examples")

# Check the class distribution for the 'train' split
print("\nClass distribution for the 'train' split:")
# The labels are 'negative', 'neutral', 'positive'
# tweet_eval sentiment labels are 0: negative, 1: neutral, 2: positive
label_names = dataset['train'].features['label'].names

for i, name in enumerate(label_names):
    count = sum(1 for label in dataset['train']['label'] if label == i)
    print(f"  Label {i} ({name}): {count} examples")

# Save two example tweets per label for later visualization
example_tweets = {}
for i, name in enumerate(label_names):
    examples = [text for text, label in zip(dataset['train']['text'], dataset['train']['label']) if label == i][:2]
    example_tweets[name] = examples

print("\nExample tweets per label (for later visualization):")
for label, tweets in example_tweets.items():
    print(f"  {label}:\n    - '{tweets[0]}'\n    - '{tweets[1]}'")

Clearing Hugging Face cache at /root/.cache/huggingface


Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]


Dataset splits distribution:
  train: 45615 examples
  test: 12284 examples
  validation: 2000 examples

Class distribution for the 'train' split:
  Label 0 (negative): 7093 examples
  Label 1 (neutral): 20673 examples
  Label 2 (positive): 17849 examples

Example tweets per label (for later visualization):
  negative:
    - 'So disappointed in wwe summerslam! I want to see john cena wins his 16th title'
    - 'That sucks if you have to take the SATs tomorrow'
  neutral:
    - '"Ben Smith / Smith (concussion) remains out of the lineup Thursday, Curtis #NHL #SJ"'
    - 'Sorry bout the stream last night I crashed out but will be on tonight for sure. Then back to Minecraft in pc tomorrow night.'
  positive:
    - '"QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"'
    - '@user Alciato: Bee will invest 150 million in January, another 200 in the Summer and plans to bring Messi by 2017"'


In [ ]:
import pandas as pd
from datasets import load_dataset

# Load the dataset
dataset = load_dataset('tweet_eval', 'sentiment')

# Get label names
label_names = dataset['train'].features['label'].names

# Calculate distribution for each split
for split in dataset.keys():
    labels = dataset[split]['label']
    df = pd.DataFrame(labels, columns=['label'])
    counts = df['label'].value_counts().sort_index()

    print(f"\nDistribution for {split} split:")
    for i, name in enumerate(label_names):
        count = counts.get(i, 0)
        percentage = (count / len(labels)) * 100
        print(f"  {name} (Label {i}): {count} ({percentage:.2f}%)")

## 2. Pipeline de tokenisation

In [1]:
import sys

# Consolidate all necessary installations to ensure compatibility and fresh loading after restart

# First, ensure system-level build tools for Rust are available
# Removed > /dev/null to see output and verify installation
!apt-get update -qq
!apt-get install -y rustc cargo

# Upgrade pip and wheel for general robustness
!pip install --upgrade pip wheel -q

# Explicitly install tokenizers with a version known to be compatible with transformers 4.38.2
# (e.g., 0.19.1 was released around the same time as transformers 4.38.x)
!pip install tokenizers==0.19.1 -q

# Then install transformers, letting it use the explicitly installed tokenizers
!pip install transformers==4.38.2 -q

# Force reinstallation of datasets, numpy, and scipy to ensure consistent versions after core HF libs
# Keep these pinned as they were identified as sources of compatibility issues before
!pip install datasets==2.16.1 numpy==1.26.4 scipy==1.12.0 --force-reinstall -q

print("Installation attempt complete. Please RESTART YOUR RUNTIME now (Runtime -> Restart runtime) for changes to take effect.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libstd-rust-1.75 libstd-rust-dev
Suggested packages:
  cargo-doc llvm-17 lld-17 clang-17
The following NEW packages will be installed:
  cargo libstd-rust-1.75 libstd-rust-dev rustc
0 upgraded, 4 newly installed, 0 to remove and 69 not upgraded.
Need to get 98.1 MB of archives.
After this operation, 392 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libstd-rust-1.75 amd64 1.75.0+dfsg0ubuntu1~bpo0-0ubuntu0.22.04.1 [46.3 MB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libstd-rust-dev amd64 1.75.0+dfsg0ubuntu1~bpo0-0ubuntu0.22.04.1 [41.6 MB]
Get:3 http://archive.ubuntu.com/ubu

### ⚠️ Action Requise : Redémarrage et Ré-exécution

Suite à l'installation des versions spécifiques de `transformers` et `huggingface_hub` ci-dessus :
1. **Redémarrez la session** : Menu `Exécution` > `Redémarrer la session`.
2. **Exécutez à nouveau les cellules** : Commencez par la cellule de tokenisation ci-dessous.

Cela corrigera l'erreur `ImportError: cannot import name 'DryRunError'` et initialisera les variables manquantes (`tokenizer`, `trainer`, etc.).

In [6]:
from transformers import AutoTokenizer

# Initialize AutoTokenizer with distilbert-base-uncased
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

# Define the preprocessing function
def preprocess_function(examples):
    # Truncate/pad tweets to 128 tokens, return input_ids, attention_mask
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

# Map the preprocessing function to the entire dataset
tokenized_dataset = dataset.map(preprocess_function, batched=True)

# Remove the original 'text' column as it's no longer needed after tokenization
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

# Rename the 'label' column to 'labels' to match the expected format for Hugging Face Trainer
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

# Set the format to "torch" (or "tf" if using TensorFlow)
tokenized_dataset.set_format("torch")

# Shuffle the training set
tokenized_dataset["train"] = tokenized_dataset["train"].shuffle(seed=42)

# Display a sample of the tokenized dataset
print("\nSample of tokenized dataset (training split):")
print(tokenized_dataset["train"][0])

print("\nDataset features after tokenization:")
print(tokenized_dataset["train"].features)

ImportError: cannot import name 'DryRunError' from 'huggingface_hub.errors' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/errors.py)

### 3. Réglage fin (Fine-tuning)
Nous allons configurer le modèle `AutoModelForSequenceClassification` avec 3 étiquettes et utiliser le `Trainer` de Hugging Face.

In [1]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# 1. Charger le modèle avec 3 labels
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=3)

# 2. Fonction pour calculer les métriques
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    return {"accuracy": acc, "f1": f1}

# 3. Arguments d'entraînement
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir='./logs',
)

# 4. Initialiser le Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
)

# Lancer l'entraînement
trainer.train()

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

## 4. Évaluation et étalonnage
Nous allons évaluer le modèle sur l'ensemble de test et visualiser la distribution de la confiance.

In [2]:
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torch

# Évaluation finale sur le test set
test_results = trainer.predict(tokenized_dataset["test"])
print(f"Test Metrics: {test_results.metrics}")

# Collecte des scores softmax
logits = torch.from_numpy(test_results.predictions)
probs = F.softmax(logits, dim=-1)
confidences, predictions = torch.max(probs, dim=-1)

# Histogramme de confiance
plt.figure(figsize=(10, 6))
plt.hist(confidences.numpy(), bins=10, range=(0, 1), edgecolor='black')
plt.title("Distribution des scores de confiance (Softmax)")
plt.xlabel("Confiance")
plt.ylabel("Nombre d'exemples")
plt.show()

NameError: name 'trainer' is not defined

## 5. Inspection d'attention
Visualisation des poids d'attention de la dernière couche pour une phrase exemple.

In [3]:
from transformers import AutoModel
import seaborn as sns

# Charger le modèle de base pour l'attention
base_model = AutoModel.from_pretrained('distilbert-base-uncased', output_attentions=True)
base_model.to(model.device)

# Sélection d'un tweet exemple (Positif)
text = "I love this new movie, it is absolutely fantastic!"
inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = base_model(**inputs)
    attentions = outputs.attentions # Liste de tenseurs (couches)

# Poids de la dernière couche, moyenne sur toutes les têtes
last_layer_attention = attentions[-1][0].mean(dim=0) # [seq_len, seq_len]

# Attention portée par le token [CLS] vers les autres tokens
cls_attention = last_layer_attention[0].cpu().numpy()
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

plt.figure(figsize=(12, 4))
sns.barplot(x=tokens, y=cls_attention)
plt.title(f"Attention du token [CLS] pour : '{text}'")
plt.xticks(rotation=45)
plt.show()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NameError: name 'tokenizer' is not defined

## 6. Fonction d'inférence explicable (Production)
Cette fonction `analyze_text` renvoie l'étiquette, la confiance et les tokens qui ont le plus contribué à la décision.

In [4]:
def analyze_text(text):
    # Préparation de l'entrée
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(model.device)

    with torch.no_grad():
        # Inférence avec le modèle de classification
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)
        confidence, pred_idx = torch.max(probs, dim=-1)

        # Récupération de l'attention via le modèle de base
        base_outputs = base_model(**inputs)
        # Moyenne des têtes de la dernière couche
        attentions = base_outputs.attentions[-1][0].mean(dim=0)
        # Attention du token [CLS] vers les autres
        cls_attn = attentions[0].cpu().numpy()

    # Reconstruction des tokens
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    # Identification des 3 tokens les plus importants (hors [CLS] / [SEP])
    important_indices = cls_attn.argsort()[-5:][::-1]
    highlights = [tokens[i] for i in important_indices if tokens[i] not in ['[CLS]', '[SEP]', '[PAD]']][:3]

    return {
        "label": label_names[pred_idx.item()],
        "confidence": f"{confidence.item():.2%}",
        "highlighted_tokens": highlights
    }

# Test de la fonction
test_tweet = "The service was slow but the food was delicious!"
result = analyze_text(test_tweet)
print(f"Texte: {test_tweet}")
print(f"Résultat: {result}")

NameError: name 'tokenizer' is not defined